# Module 9 · 音訊 1：經典音訊特徵（快速帶過）

> **本節定位｜快速帶過（經典基礎）**
> MFCC 與手工頻譜特徵（spectral centroid/rolloff/ZCR）是語音/音樂處理的經典特徵，
> 至今仍用於輕量任務。但 **2026 主流是把音訊轉成 log-mel 頻譜，餵給
> Whisper / wav2vec2 等預訓練模型**（下一節起）。本節快速建立直覺即可。

## 學習目標
- 理解音訊的本質：**一維波形 `(samples,)`**，由取樣率 (sample rate) 決定時間解析度。
- 用 FFT 看頻譜，認識「時間–頻率」表示（spectrogram）。
- 認識 **MFCC** 與常見**手工頻譜特徵**，理解它們作為固定長度特徵的用途與限制。

In [1]:
import numpy as np

# 合成一段音訊（無需下載）：440Hz + 880Hz 兩個正弦波疊加，取樣率 16kHz，1 秒
sr = 16000
t = np.linspace(0, 1.0, sr, endpoint=False)
wave = 0.6*np.sin(2*np.pi*440*t) + 0.4*np.sin(2*np.pi*880*t)
wave = wave.astype(np.float32)
print(f"波形形狀 = {wave.shape}  (samples,)  ← 一維訊號")
print(f"取樣率 sr = {sr} Hz；時長 = {len(wave)/sr:.1f} 秒")

波形形狀 = (16000,)  (samples,)  ← 一維訊號
取樣率 sr = 16000 Hz；時長 = 1.0 秒


## 1. 從波形到頻譜：短時傅立葉 (STFT)

把波形切成許多重疊小窗，每窗做 FFT，得到「**時間 × 頻率**」的能量圖（spectrogram）。
這裡用 numpy 做一個簡化版（完全離線）。

In [2]:
def simple_spectrogram(x, n_fft=512, hop=256):
    frames = []
    window = np.hanning(n_fft)
    for start in range(0, len(x) - n_fft, hop):
        seg = x[start:start+n_fft] * window
        mag = np.abs(np.fft.rfft(seg))
        frames.append(mag)
    return np.array(frames).T            # (freq_bins, time_frames)

spec = simple_spectrogram(wave)
print(f"頻譜圖形狀 = {spec.shape}  (freq_bins, time_frames)")
peak_bin = spec.mean(axis=1).argmax()
peak_hz = peak_bin * sr / 512
print(f"能量最強的頻率 bin ≈ {peak_hz:.0f} Hz（應接近我們放入的 440Hz 主頻）")

頻譜圖形狀 = (257, 61)  (freq_bins, time_frames)
能量最強的頻率 bin ≈ 438 Hz（應接近我們放入的 440Hz 主頻）


## 2. MFCC 與手工頻譜特徵（需 librosa）

MFCC（梅爾頻率倒譜係數）模擬人耳對頻率的非線性感知，是語音的經典特徵；
配合 spectral centroid（亮度）、rolloff、zero-crossing rate（粗糙度）等描述音色。
需 `multimodal` extra：`uv sync --extra multimodal`。

In [3]:
try:
    import librosa
    mfcc = librosa.feature.mfcc(y=wave, sr=sr, n_mfcc=13)         # (13, T)
    centroid = librosa.feature.spectral_centroid(y=wave, sr=sr)  # (1, T)
    zcr = librosa.feature.zero_crossing_rate(wave)               # (1, T)
    print(f"MFCC 形狀          = {mfcc.shape}  (n_mfcc, T)")
    print(f"spectral centroid  = {centroid.shape}  平均 {centroid.mean():.0f} Hz")
    print(f"zero-crossing rate = {zcr.shape}  平均 {zcr.mean():.3f}")
    # 把時間軸聚合（取平均）→ 固定長度特徵向量，供傳統分類器使用
    feat_vec = np.concatenate([mfcc.mean(axis=1), centroid.mean(axis=1), zcr.mean(axis=1)])
    print(f"\n聚合後固定長度特徵向量維度 = {feat_vec.shape}")
except Exception as e:
    print("（未安裝 librosa，略過實際計算）", e)
    print("概念：MFCC≈13 維/幀，描述音色；對時間軸取平均可得固定長度特徵，餵傳統分類器。")

（未安裝 librosa，略過實際計算） No module named 'librosa'
概念：MFCC≈13 維/幀，描述音色；對時間軸取平均可得固定長度特徵，餵傳統分類器。


## 3. 限制 → 為何 2026 改用預訓練音訊模型

| 面向 | 手工特徵（本節） | 預訓練模型（下一節 Whisper/wav2vec2） |
|:--|:--|:--|
| 特徵 | 人工設計（MFCC/頻譜統計） | 在海量音訊上自學的表示 |
| 語意 | 低（聲學統計） | 高（能辨識語音內容、語者、事件） |
| 任務 | 需逐任務調特徵 | 一個模型遷移到 ASR/分類/情緒等 |

**結論**：手工特徵幫助理解「波形→特徵」，但 2026 主流是先把音訊標準化成
**log-mel 頻譜**（下一節），再交給預訓練模型。

---
### 小結
- 音訊 = 一維波形 `(samples,)`，由取樣率決定。
- STFT → 時間×頻率頻譜；MFCC/頻譜統計可聚合成固定長度特徵。
- 手工特徵無語意 → 被預訓練音訊模型取代。